In [10]:
import cv2
import numpy as np
import json
import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go

from IMMTracker import *

from ultralytics import YOLO
import albumentations as A
import torch
import torchvision
import torchaudio

In [11]:
print("Активні моделі IMM:")
for fname in ("fx_ballistic", "fx_hit", "fx_bounce", "hx",
              "measurement_transform", "get_dynamic_transition_matrix"):
    obj = globals().get(fname)
    print(f"  {fname:30s} -> {getattr(obj, '__module__', 'N/A')}.{fname}")


Активні моделі IMM:
  fx_ballistic                   -> IMM_UKF.fx_ballistic
  fx_hit                         -> IMM_UKF.fx_hit
  fx_bounce                      -> IMM_UKF.fx_bounce
  hx                             -> IMM_UKF.hx
  measurement_transform          -> IMM_UKF.measurement_transform
  get_dynamic_transition_matrix  -> IMM_UKF.get_dynamic_transition_matrix


In [12]:
# Реперні точки кутів волейбольного майданчика. Конвенція системи світу:
#   X — поперек майданчика (вздовж лицевої лінії),
#   Y — вертикальна вісь (вгору, перпендикулярно підлозі),
#   Z — вздовж довгої сторони (вздовж бічної лінії).
# Підлога відповідає Y = 0; на підлозі лежать усі 4 реперні кути.
pts_real_3d = np.array([
    [0.0, 0.0,  0.0],
    [9.0, 0.0,  0.0],
    [9.0, 0.0, 18.0],
    [0.0, 0.0, 18.0]], dtype=np.float32)

# Піксельні координати тих самих кутів у кадрі (порядок такий самий, як у pts_real_3d).
pts_video_2d = np.array([
    [73,   1031],
    [1835, 1027],
    [1504,  606],
    [405,   606]], dtype=np.float32)

# Внутрішні параметри камери (фокальна 1300 px, оптичний центр у середині FullHD-кадру).
K = np.array([
    [1300.0,    0.0,  960.0],
    [   0.0, 1300.0,  540.0],
    [   0.0,    0.0,    1.0]], dtype=np.float32)

R, tvec, camera_pos = calibrate_camera(pts_real_3d, pts_video_2d, K, dist=np.zeros((4, 1)))

# Спільні параметри
frames_per_second = 50
dt = 1 / frames_per_second  # 0.02 секунди між кадрами при 50 FPS
dim_x = 6
dim_z = 3
points = MerweScaledSigmaPoints(n=dim_x, alpha=.1, beta=2., kappa=1.)
P_init = np.diag([0.1, 50.0, 0.1, 50.0, 0.1, 50.0])
R_init = np.diag([0.01, 0.01, 0.04])

# Для балістичної моделі Q мінімальна.
q_var_ballistic = 0.1
q_b = Q_discrete_white_noise(dim=2, dt=dt, var=q_var_ballistic)
Q_ballistic = block_diag(q_b, q_b, q_b)

# Для моделі удару Q велика.
q_var_hit = 100
q_h = Q_discrete_white_noise(dim=2, dt=dt, var=q_var_hit)
Q_hit = block_diag(q_h, q_h, q_h)

# Для відскоку Q середня.
q_var_bounce = 5
q_bnc = Q_discrete_white_noise(dim=2, dt=dt, var=q_var_bounce)
Q_bounce = block_diag(q_bnc, q_bnc, q_bnc)


In [13]:
path_to_load = 'detect_infos/detection_data.json'
with open(path_to_load, 'r', encoding='utf-8') as f:
    raw_detections = json.load(f)

detections_df = pd.DataFrame.from_dict(raw_detections)
detections_df.set_index('frame', inplace=True)
detections_df["ball_detected"] = detections_df["ball_detected"].fillna(False)

In [14]:
# main_model = YOLO('/home/var-roman/Desktop/my_projects/diploma_project/training_models/models/main_model_april.pt')
#
# results = main_model.predict(
#     source='/home/var-roman/Desktop/my_projects/diploma_project/data/videos/Japan_vs_Poland_ultrashort.mp4',
#     imgsz=1920,
#     conf=0.4,
#     iou=0.45,
#     save=False,
#     stream=True
# )
#
# raw_detections = []
# frame_count = 0
#
# for r in results:
#     frame_count += 1
#
#     if len(r.boxes) > 0:
#         box = r.boxes[0].xywh[0]
#         raw_detections.append({
#             'ball_detected': True,
#             'frame': frame_count,
#             'x_pos': float(box[0]),
#             'y_pos': float(box[1]),
#             'w_box': float(box[2]),
#         })
#     else:
#         raw_detections.append({'ball_detected': False, 'frame': frame_count})
#
# path_to_save = 'detect_infos/raw_detection_data.json'
# with open(path_to_save, 'w', encoding='utf-8') as f:
#     json.dump(raw_detections, f, ensure_ascii=False, indent=4)

In [15]:
detections_df

,ball_detected,x_pos,y_pos,w_box
frame,,,,
1,False,NaN,NaN,NaN
2,False,NaN,NaN,NaN
3,False,NaN,NaN,NaN
4,False,NaN,NaN,NaN
5,False,NaN,NaN,NaN
...,...,...,...,...
1131,False,NaN,NaN,NaN
1132,False,NaN,NaN,NaN
1133,False,NaN,NaN,NaN


In [16]:
# Старий код для перевірки роботи однієї моделі UKF(балістичної) у вигляді тесту та виправлення помилок
# UKF = UnscentedKalmanFilter(dim_x=dim_x, dim_z=dim_z, dt=dt, fx=fx_ballistic, hx=hx, points=points)
# UKF.P = np.diag([0.1, 50.0, 0.1, 50.0, 0.1, 50.0])
# UKF.Q = Q_ballistic
# UKF.R = np.diag([0.01, 0.01, 0.04])
#
# is_initialized = False
# smoothed_x, smoothed_y, smoothed_z = [], [], []
#
# for i in detections_df.iterrows():
#     if not is_initialized:
#         if i[1]['ball_detected']:
#             z = measurement_transform(i[1], K, R, camera_pos)
#             UKF.x = np.zeros([6])
#             UKF.x[0], UKF.x[2], UKF.x[4] = z
#             is_initialized = True
#         else:
#             continue
#
#     UKF.predict()
#
#     if i[1]['ball_detected']:
#         z = measurement_transform(i[1], K, R, camera_pos)
#         UKF.update(z)
#     else:
#         UKF.update(None)
#
#     smoothed_x.append(UKF.x[0]), smoothed_y.append(UKF.x[2]), smoothed_z.append(UKF.x[4])

In [17]:
# # Ballistic filter
# ukf_ballistic = UnscentedKalmanFilter(name='ballistic UKF', dim_x=dim_x, dim_z=dim_z, dt=dt, fx=fx_ballistic, hx=hx, points=points)
# ukf_ballistic.P = P_init
# ukf_ballistic.Q = Q_ballistic
# ukf_ballistic.R = R_init
#
# # Hit filter
# ukf_hit = UnscentedKalmanFilter(name='hit UKF', dim_x=dim_x, dim_z=dim_z, dt=dt, fx=fx_hit, hx=hx, points=points)
# ukf_hit.P = P_init
# ukf_hit.Q = Q_hit
# ukf_hit.R = R_init
#
# # Bounce filter
# ukf_bounce = UnscentedKalmanFilter(name='bounce UKF', dim_x=dim_x, dim_z=dim_z, dt=dt, fx=fx_bounce, hx=hx, points=points)
# ukf_bounce.P = P_init
# ukf_bounce.Q = Q_bounce
# ukf_bounce.R = R_init
#
# filters_lt = [ukf_ballistic, ukf_hit, ukf_bounce]
# mu = np.array([0.95, 0.04, 0.01])
# M_base = np.array([[0.95, 0.04, 0.01],
#                    [0.60, 0.40, 0.00],
#                    [0.90, 0.00, 0.10]])
#
# imm_base = IMMEstimator(filters_lt, mu, M_base)
#
# is_initialized = False
# smoothed_x, smoothed_y, smoothed_z = [], [], []
# h_matrix = np.array([
#     [1, 0, 0, 0, 0, 0],
#     [0, 0, 1, 0, 0, 0],
#     [0, 0, 0, 0, 1, 0]
# ], dtype=float)
#
# for i, row in detections_df.iterrows():
#     if not is_initialized:
#         if row['ball_detected']:
#             z = measurement_transform(row, K, R, camera_pos)
#             imm_base.x[0], imm_base.x[2], imm_base.x[4] = z
#             imm_base.filter_setx()
#             is_initialized = True
#         else:
#             smoothed_x.append(None), smoothed_y.append(None), smoothed_z.append(None)
#             continue
#
#     imm_base.predict()
#
#     if row['ball_detected']:
#         z = measurement_transform(row, K, R, camera_pos)
#
#         z_mean = np.dot(h_matrix, imm_base.x_prior)
#         S = np.dot(h_matrix, np.dot(imm_base.P_prior, h_matrix.T)) + np.diag([0.01, 0.01, 0.04]) # R матриця
#         y = z - z_mean
#
#         try:
#             S_inv = np.linalg.inv(S)
#             mahal_dist_sq = np.dot(y.T, np.dot(S_inv, y))
#         except np.linalg.LinAlgError:
#             mahal_dist_sq = 1e5
#
#         if mahal_dist_sq > 11.34:
#             imm_base.update(None)
#         else:
#             imm_base.update(z)
#     else:
#         imm_base.update(None)
#
#     smoothed_x.append(imm_base.x[0])
#     smoothed_y.append(imm_base.x[2])
#     smoothed_z.append(imm_base.x[4])
#
# detections_df[['smoothed_x', 'smoothed_y', 'smoothed_z']] = np.nan, np.nan, np.nan
# smoothed_values = np.column_stack([smoothed_x, smoothed_y, smoothed_z])
# detections_df[['smoothed_x', 'smoothed_y', 'smoothed_z']] = smoothed_values

In [18]:
# КРИТИЧНО: IMMTracker очікує dt у секундах між кадрами (1/FPS), не FPS!
# Попередня версія передавала dt=frames_per_second=50, що означало
# фізичну інтеграцію з кроком 50 секунд між кадрами та повну деградацію
# фільтра. Тут зафіксовано як 1.0/frames_per_second.
tracker = IMMTracker(dt=1.0/frames_per_second, max_age=50, min_hits=3)
final_trajectory = []

for frame_idx, row in detections_df.iterrows():
    # Для волейбольного відео в одному кадрі може бути лише одна детекція м'яча
    # (детектор повертає не більше одного об'єкта найвищої впевненості).
    current_detections = []

    if row['ball_detected']:
        z = measurement_transform(row, K, R, camera_pos)
        # Запасний фільтр явно нефізичних детекцій нижче підлоги
        # (припустимий від'ємний запас 0.5 м для шуму глибини).
        if z[1] >= -0.5:
            current_detections.append(z)

    # Трекер сам коректно обробляє порожній список детекцій — викликає
    # mark_missed() усім активним трекам та керує життєвим циклом.
    tracker.update(current_detections)

    confirmed = tracker.get_confirmed_tracks()
    if len(confirmed) > 0:
        # Обираємо найдовший підтверджений трек (трек із найбільшою кількістю
        # успішних оновлень — імовірний цільовий м'яч).
        best_track = max(confirmed, key=lambda t: t.hits)
        final_trajectory.append({
            'frame':      frame_idx,
            'smoothed_x': best_track.imm.x[0],
            'smoothed_y': best_track.imm.x[2],
            'smoothed_z': best_track.imm.x[4],
            'track_id':   best_track.track_id,
        })
    else:
        final_trajectory.append({
            'frame':      frame_idx,
            'smoothed_x': None,
            'smoothed_y': None,
            'smoothed_z': None,
            'track_id':   None,
        })


In [19]:
full_detections_df = detections_df.join(pd.DataFrame(final_trajectory).set_index('frame'), on='frame', how='left')

In [20]:
full_detections_df

,ball_detected,x_pos,y_pos,w_box,smoothed_x,smoothed_y,smoothed_z,track_id
frame,,,,,,,,
1,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,False,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...
1131,False,NaN,NaN,NaN,7.206758,7.084286,3.040179,35.0
1132,False,NaN,NaN,NaN,7.207173,7.118620,3.117185,35.0
1133,False,NaN,NaN,NaN,7.207586,7.148668,3.193830,35.0


In [21]:
smoothed_x, smoothed_y, smoothed_z = full_detections_df[['smoothed_x', 'smoothed_y', 'smoothed_z']].values.reshape((3, len(full_detections_df)))
fig = px.line_3d(x=smoothed_x, y=smoothed_z, z=smoothed_y, width=500, height=500)
fig.show()